# Real-Time Fraud Detection with SparkRules

Build a fraud detection pipeline that:
1. Evaluates transactions against velocity, amount, and device rules
2. Uses `stop_on_fire` to immediately block high-risk transactions
3. Generates risk scores and action recommendations
4. Exports rules to OPA Rego for security team review

```bash
pip install sparkrules
```

In [ ]:
from sparkrules.executor.local_executor import LocalRuleExecutor
from sparkrules.export.opa import export_to_rego
import json

## Define fraud detection rules

Key patterns:
- **stop_on_fire**: when a BLOCK rule fires, skip all lower-priority rules
- **reason_codes**: attach fraud codes for investigation tracking
- **salience**: highest-risk rules evaluated first

In [ ]:
FRAUD_RULES = """
rule "block-velocity-spike"
  salience 100
  reason_codes ["FR001", "VELOCITY"]
  when $txn : Txn( $txn.count_24h > 20 )
  then result.action = "BLOCK"; result.risk = 95; result.reason = "Velocity spike"; end

rule "block-large-amount"
  salience 90
  reason_codes ["FR002", "AMOUNT"]
  when $txn : Txn( $txn.amount > 10000 )
  then result.action = "BLOCK"; result.risk = 90; result.reason = "Large amount"; end

rule "hold-geo-risk"
  salience 70
  reason_codes ["FR003"]
  when $txn : Txn( $txn.geo_risk > 0.8 )
  then result.action = "HOLD"; result.risk = 75; end

rule "review-new-device"
  salience 60
  reason_codes ["FR004"]
  when $txn : Txn( $txn.device_age_days < 3 )
  then result.action = "REVIEW"; result.risk = 60; end

rule "review-first-merchant"
  salience 50
  reason_codes ["FR005"]
  when $txn : Txn( $txn.first_merchant == true )
  then result.action = "REVIEW"; result.risk = 50; end

rule "allow-trusted"
  salience 10
  when $txn : Txn( $txn.trusted == true )
  then result.action = "ALLOW"; result.risk = 5; end
"""

executor = LocalRuleExecutor.from_drl(FRAUD_RULES)
print(f"Loaded {len(executor.rulepack.rules)} fraud rules")
print(f"Alpha network: {executor.alpha_net.unique_alphas} unique predicates")

## Evaluate a batch of transactions

In [ ]:
transactions = [
    {
        "id": "TXN-001",
        "txn": {
            "amount": 150,
            "count_24h": 3,
            "geo_risk": 0.1,
            "device_age_days": 365,
            "first_merchant": False,
            "trusted": True,
        },
    },
    {
        "id": "TXN-002",
        "txn": {
            "amount": 15000,
            "count_24h": 2,
            "geo_risk": 0.3,
            "device_age_days": 90,
            "first_merchant": False,
            "trusted": False,
        },
    },
    {
        "id": "TXN-003",
        "txn": {
            "amount": 500,
            "count_24h": 25,
            "geo_risk": 0.9,
            "device_age_days": 1,
            "first_merchant": True,
            "trusted": False,
        },
    },
    {
        "id": "TXN-004",
        "txn": {
            "amount": 200,
            "count_24h": 5,
            "geo_risk": 0.2,
            "device_age_days": 2,
            "first_merchant": False,
            "trusted": False,
        },
    },
    {
        "id": "TXN-005",
        "txn": {
            "amount": 3000,
            "count_24h": 8,
            "geo_risk": 0.85,
            "device_age_days": 30,
            "first_merchant": True,
            "trusted": False,
        },
    },
]

print(f"{'TXN':<10s} {'Amount':>8s} {'Action':<8s} {'Risk':>4s} {'Rules Fired'}")
print("-" * 60)
for txn in transactions:
    result = executor.score(txn)
    action = result.merged_actions.get("action", "NONE")
    risk = result.merged_actions.get("risk", 0)
    fired = [f.rule_name for f in result.fires if f.fired]
    print(f"{txn['id']:<10s} ${txn['txn']['amount']:>7,d} {action:<8s} {risk:>4} {fired}")

## Export rules to OPA Rego for security review

In [ ]:
rego = export_to_rego(FRAUD_RULES, package_name="fraud.authorization")
print(rego)